# Visual Feature Extraction — IEMOCAP

All three visual streams share the same face-crop pipeline:
1. Sample frames at 2 fps (max 60)
2. RetinaFace detects face → crops once per frame
3. Valid crops split into temporal segments: beginning / middle / end (3 frames each)
4. All three models receive the same face crops

**Outputs:** `Dataset/Processed/IEMOCAP/features/`
- `video_clip_temporal.pt`    — `{utt_id: np.array(3, 768)}`   CLIP ViT-L/14
- `video_siglip2_temporal.pt` — `{utt_id: np.array(3, 1152)}`  SigLIP 2
- `video_openface_au.pt`      — `{utt_id: np.array(3, 8)}`     AU intensities (temporal)

IEMOCAP videos are speaker-cropped — expect >95% face coverage.

In [ ]:
import os
import cv2
import tempfile
import torch
import numpy as np
import pandas as pd
import clip
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModel as HFAutoModel
from openface.face_detection import FaceDetector
from openface.multitask_model import MultitaskPredictor

PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/IEMOCAP"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

FRAME_SAMPLE_FPS  = 2
MAX_FRAMES        = 60
N_FRAMES_PER_SEG  = 3
AU_CONF_THRESHOLD = 0.5
AU_DIM            = 8
CKPT_EVERY        = 200    # save checkpoint every N utterances
OF3_WEIGHTS_DIR   = Path.home() / ".cache" / "openface"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances: {len(labels)}")

In [2]:
def sample_frames(video_path: Path, fps=FRAME_SAMPLE_FPS, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    src_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(src_fps / fps)))
    frames, idx = [], 0
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret or frame is None or frame.size == 0:
            break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def temporal_segments(items: list, n: int = N_FRAMES_PER_SEG) -> tuple:
    total  = len(items)
    mid    = total // 2
    half_n = n // 2
    beg     = items[:n]
    mid_seg = items[max(0, mid - half_n): max(0, mid - half_n) + n]
    end     = items[max(0, total - n):]
    return beg, mid_seg, end

## Load Models

In [3]:
clip_model, clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
CLIP_DIM = clip_model.visual.output_dim
print(f"CLIP        : ViT-L/14  dim={CLIP_DIM}")

CLIP        : ViT-L/14  dim=768


In [4]:
siglip2_processor = AutoProcessor.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model     = HFAutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model.eval()
for p in siglip2_model.parameters():
    p.requires_grad_(False)
siglip2_model = siglip2_model.to(DEVICE)
SIGLIP2_DIM   = siglip2_model.config.vision_config.hidden_size
print(f"SigLIP 2    : so400m-patch14-384  dim={SIGLIP2_DIM}")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

SigLIP 2    : so400m-patch14-384  dim=1152


In [5]:
of3_face = FaceDetector(
    model_path=str(OF3_WEIGHTS_DIR / "Alignment_RetinaFace.pth"),
    device=str(DEVICE), vis_threshold=AU_CONF_THRESHOLD,
)
of3_pred = MultitaskPredictor(
    model_path=str(OF3_WEIGHTS_DIR / "MTL_backbone.pth"),
    device=str(DEVICE),
)
print(f"OpenFace 3.0: RetinaFace + MTL backbone  AU_DIM={AU_DIM}")

Loading pretrained model from /home/jubaer/.cache/openface/Alignment_RetinaFace.pth
remove prefix 'module.'
Missing keys:0
Unused checkpoint keys:0
Used keys:300
Loading multitask model from /home/jubaer/.cache/openface/MTL_backbone.pth...


/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


OpenFace 3.0: RetinaFace + MTL backbone  AU_DIM=8


## Extraction Functions

In [6]:
def collect_face_crops(video_path: Path) -> list[np.ndarray]:
    frames = sample_frames(video_path)
    crops  = []
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, frame in enumerate(frames):
            fpath = os.path.join(tmpdir, f"f{i:04d}.jpg")
            cv2.imwrite(fpath, frame)
            crop, _ = of3_face.get_face(fpath)
            if crop is not None and crop.size > 0:
                crops.append(crop)
    return crops

def encode_clip(crops: list[np.ndarray]) -> np.ndarray:
    if not crops:
        return np.zeros((3, CLIP_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(CLIP_DIM, dtype=np.float32)); continue
        pil = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in seg]
        t   = torch.stack([clip_preprocess(p) for p in pil]).to(DEVICE)
        with torch.no_grad():
            e = clip_model.encode_image(t).float()
            e = e / e.norm(dim=-1, keepdim=True)
        vecs.append(e.cpu().numpy().mean(axis=0))
    return np.stack(vecs)

def encode_siglip2(crops: list[np.ndarray]) -> np.ndarray:
    if not crops:
        return np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(SIGLIP2_DIM, dtype=np.float32)); continue
        pil    = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in seg]
        inputs = siglip2_processor(images=pil, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = siglip2_model.vision_model(**inputs)
        vecs.append(out.pooler_output.cpu().float().numpy().mean(axis=0))
    return np.stack(vecs)

def encode_au(crops: list[np.ndarray]) -> np.ndarray:
    if not crops:
        return np.zeros((3, AU_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(AU_DIM, dtype=np.float32)); continue
        aus = []
        for crop in seg:
            _, _, au_out = of3_pred.predict(crop)
            aus.append(au_out.cpu().float().numpy().flatten())
        vecs.append(np.stack(aus).mean(axis=0))
    return np.stack(vecs)

print("Extraction functions defined.")

Extraction functions defined.


## Extraction — Unified Loop

> **Re-run note:** existing `.pt` files used full-frame CLIP/SigLIP2 and flat AU.
> Run the delete cell first, then the extraction cell.

In [ ]:
# Delete old files — run once before the extraction loop
# for tag in ["video_clip_temporal", "video_siglip2_temporal", "video_openface_au"]:
#     p = FEATURES_DIR / f"{tag}.pt"
#     if p.exists():
#         p.unlink()
#         print(f"Deleted: {p.name}")
# print("Done.")

Done.


In [ ]:
clip_path = FEATURES_DIR / "video_clip_temporal.pt"
sig_path  = FEATURES_DIR / "video_siglip2_temporal.pt"
au_path   = FEATURES_DIR / "video_openface_au.pt"

if clip_path.exists() and sig_path.exists() and au_path.exists():
    print("All exist — skipping")
else:
    # Checkpoint files — deleted on successful completion
    ckpt_clip = FEATURES_DIR / "video_clip_temporal_ckpt.pt"
    ckpt_sig  = FEATURES_DIR / "video_siglip2_temporal_ckpt.pt"
    ckpt_au   = FEATURES_DIR / "video_openface_au_ckpt.pt"

    # Resume from checkpoint if exists, else start fresh
    clip_feats: dict[str, np.ndarray] = torch.load(ckpt_clip, weights_only=False) if ckpt_clip.exists() else {}
    sig_feats:  dict[str, np.ndarray] = torch.load(ckpt_sig,  weights_only=False) if ckpt_sig.exists()  else {}
    au_feats:   dict[str, np.ndarray] = torch.load(ckpt_au,   weights_only=False) if ckpt_au.exists()   else {}
    already_done = set(clip_feats.keys())

    n_resume = len(already_done)
    print(f"IEMOCAP visual  ({len(labels)} utterances, {n_resume} already done)")

    skipped:   list[str] = []
    zero_face: list[str] = []
    n_new = 0

    for _, row in tqdm(labels.iterrows(), total=len(labels), desc="IEMOCAP visual"):
        if row["utt_id"] in already_done:
            continue

        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists():
            skipped.append(row["utt_id"]); continue

        crops = collect_face_crops(vid_path)

        if not crops:
            zero_face.append(row["utt_id"])
            clip_feats[row["utt_id"]] = np.zeros((3, CLIP_DIM),    dtype=np.float32)
            sig_feats [row["utt_id"]] = np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
            au_feats  [row["utt_id"]] = np.zeros((3, AU_DIM),      dtype=np.float32)
        else:
            clip_feats[row["utt_id"]] = encode_clip(crops)
            sig_feats [row["utt_id"]] = encode_siglip2(crops)
            au_feats  [row["utt_id"]] = encode_au(crops)

        n_new += 1
        if n_new % CKPT_EVERY == 0:
            torch.save(clip_feats, ckpt_clip)
            torch.save(sig_feats,  ckpt_sig)
            torch.save(au_feats,   ckpt_au)
            print(f"  [ckpt] {len(clip_feats)} total  ({n_new} new this run)")

    # Save final files and remove checkpoints
    torch.save(clip_feats, clip_path)
    torch.save(sig_feats,  sig_path)
    torch.save(au_feats,   au_path)
    for ckpt in [ckpt_clip, ckpt_sig, ckpt_au]:
        if ckpt.exists():
            ckpt.unlink()

    n_ok = sum(1 for v in clip_feats.values() if np.abs(np.array(v)).sum() > 1e-6)
    print(f"Saved: {len(clip_feats)} entries")
    print(f"Face coverage : {n_ok}/{len(clip_feats)}  ({n_ok/max(len(clip_feats),1)*100:.1f}%)")
    if skipped:   print(f"Skipped (missing): {len(skipped)}")
    if zero_face: print(f"No face detected : {len(zero_face)}")

In [9]:
print("=== Verification ===")
for tag, expected_shape in [
    ("video_clip_temporal",    (3, CLIP_DIM)),
    ("video_siglip2_temporal", (3, SIGLIP2_DIM)),
    ("video_openface_au",      (3, AU_DIM)),
]:
    pt = FEATURES_DIR / f"{tag}.pt"
    if not pt.exists():
        print(f"  {tag}: MISSING"); continue
    d   = torch.load(pt, weights_only=False)
    arr = np.stack([np.array(v) for v in d.values()])
    shape = np.array(next(iter(d.values()))).shape
    ok    = shape == expected_shape
    print(f"  {tag}: count={len(d)}/10039  shape={shape} "
          f"{'✓' if ok else '✗ expected '+str(expected_shape)}  "
          f"nan={np.isnan(arr).any()}  inf={np.isinf(arr).any()}")

=== Verification ===
  video_clip_temporal: count=10039/10039  shape=(3, 768) ✓  nan=False  inf=False
  video_siglip2_temporal: count=10039/10039  shape=(3, 1152) ✓  nan=False  inf=False
  video_openface_au: count=10039/10039  shape=(3, 8) ✓  nan=False  inf=False
